In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BUGGY_DATA_FILE = "buggy.csv"
PATCHED_DATA_FILE = "patched.csv"

OPERATION = "operation"
OPERATION_TARGET = "SCAN"
TIMESTAMP = "timestamp_us"
LATENCY = "latency_us"

In [ ]:
buggy_data = pd.read_csv(BUGGY_DATA_FILE)
buggy_data = buggy_data[buggy_data[OPERATION] == OPERATION_TARGET]

patched_data = pd.read_csv(PATCHED_DATA_FILE)
patched_data = patched_data[patched_data[OPERATION] == OPERATION_TARGET]

In [ ]:
BUCKET_SIZE = 20

buggy_averaged = buggy_data.groupby(buggy_data.index // BUCKET_SIZE).agg({
    LATENCY: "mean",
    TIMESTAMP: "min",
    OPERATION: "first",
})
buggy_averaged[TIMESTAMP] -= buggy_averaged[TIMESTAMP].min()  # have timestamps start at 0
buggy_averaged[TIMESTAMP] /= 1e6  # convert us to s
# drop the first few datapoints, which are usually pretty extreme
buggy_averaged = buggy_averaged.iloc[10:]

patched_averaged = patched_data.groupby(patched_data.index // BUCKET_SIZE).agg({
    LATENCY: "mean",
    TIMESTAMP: "min",
    OPERATION: "first",
})
patched_averaged[TIMESTAMP] -= patched_averaged[TIMESTAMP].min()
patched_averaged[TIMESTAMP] /= 1e6
patched_averaged = patched_averaged.iloc[10:]

In [ ]:
def timeseries(data, title):
    plt.figure(figsize=(15, 6))
    plot = sns.lineplot(data=data, x=TIMESTAMP, y=LATENCY)
    plot.set_xlabel("Time (s)")
    plot.set_ylabel("Latency (us)")
    plt.title(title)
    plt.show()

def histogram(data, title, hue, bins):
    plt.figure(figsize=(15, 6))
    plot = sns.histplot(data=data, x=LATENCY, hue=hue, bins=bins)
    plot.set_xlabel("Latency (us)")
    plt.title(title)
    plt.show()

In [ ]:
timeseries(buggy_averaged, "Query Latency for Buggy MongoDB (20-point Moving Average)")
timeseries(patched_averaged, "Query Latency for Patched MongoDB (20-point Moving Average)")

In [ ]:
# skip first 60% of data
buggy_zoomed = buggy_averaged.iloc[len(buggy_averaged) * 60 // 100 :]
patched_zoomed = patched_averaged.iloc[len(patched_averaged) * 60 // 100 :]

timeseries(buggy_zoomed, "Query Latency for Buggy MongoDB (discarding first 60% of data)")
timeseries(patched_zoomed, "Query Latency for Patched MongoDB (discarding first 60% of data)")

print("Buggy summary statistics:")
print(buggy_zoomed.describe())
print("\n\nPatched summary statistics:")
print(patched_zoomed.describe())

In [ ]:
buggy_zoomed["version"] = "buggy"
patched_zoomed["version"] = "patched"
combined = pd.concat([buggy_zoomed, patched_zoomed])
histogram(combined, "Latency Histogram for Buggy vs Patched MongoDB (discarding first 60% of data)", hue="version", bins=35)